## Definição da linguagem

In [23]:
language = 'pt'

## Instaláveis necessários para inicialização do programa

In [24]:
!pip install git+https://github.com/openai/whisper.git -q
!pip install openai
!pip install gTTS

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## Import's usados

In [25]:
from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode
import whisper
from gtts import gTTS
from openai import OpenAI

## Gravação de voz com JS e Python

In [26]:
RECORD = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({audio:true})
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async () => {
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""



def record(sec=15):
    display(Javascript(RECORD))
    js_result = output.eval_js('record(%s)' % (sec * 1000))

    # Decodifica o arquivo base64 retornado pelo JavaScript
    audio = b64decode(js_result.split(',')[1])

    # Salva o arquivo em formato .wav
    arquivo_de_voz = "request_audio.wav"
    with open(arquivo_de_voz, 'wb') as f:
        f.write(audio)

    return f'/content/{arquivo_de_voz}'


## Reconhecimento de voz com o Whisper

In [27]:
def whisper_traducao(arquivo_de_voz):
    model_whisper = whisper.load_model("small")
    result = model_whisper.transcribe(arquivo_de_voz, fp16 = False, language = language)
    # Retorna apenas o texto da transcrição
    return result["text"]


## Integração com CHATGPT

In [28]:


api_key = ""
def chatgpt_response (api_key, trascription):
  if api_key == "":
    api_key = input("Digite sua chave da OpenAI(ou copie e cole): ")
  client = OpenAI(api_key=api_key)

  response = client.chat.completions.create(
      model="gpt-3.5-turbo",
      messages=[{"role": "user", "content": trascription}]
  )

  response_final = response.choices[0].message.content
  return response_final



## Síntese de resposta com gTTS

In [29]:
def gtss_response(definition_response):
  gtts_object = gTTS(text=definition_response, lang = language, slow = False)
  response_audio = "/content/request_audio.wav"
  gtts_object.save(response_audio)
  return display(Audio(response_audio, autoplay=True))

## Dicionário usado

In [30]:
estoque_loja = {
    "PRD001": {"nome": "Notebook Core i7", "preco": 4500.00, "estoque": 12},
    "PRD002": {"nome": "Mouse Sem Fio", "preco": 85.50, "estoque": 50},
    "PRD003": {"nome": "Teclado Mecânico", "preco": 320.00, "estoque": 35},
    "PRD004": {"nome": "Monitor 24 Polegadas", "preco": 950.00, "estoque": 20},
    "PRD005": {"nome": "Cadeira Gamer", "preco": 1250.00, "estoque": 8},
    "PRD006": {"nome": "Headset Bluetooth", "preco": 280.00, "estoque": 45},
    "PRD007": {"nome": "Webcam 1080p", "preco": 199.90, "estoque": 30},
    "PRD008": {"nome": "Microfone Condensador", "preco": 450.00, "estoque": 15},
    "PRD009": {"nome": "Mesa Digitalizadora", "preco": 380.00, "estoque": 22},
    "PRD010": {"nome": "Gabinete ATX", "preco": 420.00, "estoque": 18},
    "PRD011": {"nome": "Placa de Vídeo RTX 4060", "preco": 2800.00, "estoque": 5},
    "PRD012": {"nome": "Processador Ryzen 5", "preco": 1100.00, "estoque": 25},
    "PRD013": {"nome": "Placa Mãe B550", "preco": 850.00, "estoque": 14},
    "PRD014": {"nome": "Memória RAM 16GB DDR4", "preco": 350.00, "estoque": 60},
    "PRD015": {"nome": "SSD NVMe 1TB", "preco": 490.00, "estoque": 40},
    "PRD016": {"nome": "HD Externo 2TB", "preco": 550.00, "estoque": 28},
    "PRD017": {"nome": "Fonte 650W 80 Plus", "preco": 480.00, "estoque": 24},
    "PRD018": {"nome": "Roteador Wi-Fi 6", "preco": 399.00, "estoque": 32},
    "PRD019": {"nome": "Cabo HDMI 2.1 (2m)", "preco": 45.00, "estoque": 100},
    "PRD020": {"nome": "Filtro de Linha 8 Tomadas", "preco": 65.00, "estoque": 75}
}

In [31]:








def estoque_solucao(estoque_loja, x):
    if x == 0:
        return "Fim da execução"
    elif x == 1:
        print("ouvindo...\n")
        record_file = record()
        display(Audio(record_file, autoplay=True))

        # 1. Pega o texto do Whisper
        texto_usuario = whisper_traducao(record_file)
        print(f"Você disse: {texto_usuario}")

        # 2. Manda para o ChatGPT (certifique-se de que api_key está preenchida)
        resposta_ia = chatgpt_response(api_key, texto_usuario)
        print(f"ChatGPT respondeu: {resposta_ia}")

        # 3. Transforma em áudio
        gtss_response(resposta_ia)







x = int(input("Digite [0] Para sair, [1] para perguntar algo"))
estoque_solucao(estoque_loja,x)

Digite [0] Para sair, [1] para perguntar algo1
ouvindo...



<IPython.core.display.Javascript object>

Você disse:  Eu tô já em elevated! Eu não semi-brazo mesmo! Eu comi um gallo no dedo nós vídeos vocês mas eu tenho que firmware


KeyboardInterrupt: Interrupted by user